# インバウンド認証

AgentCore Identity を使用すると、AgentCore Runtime 内のエージェントやツールを呼び出すユーザーやアプリケーションのインバウンドアクセス（インバウンド認証）を検証したり、AgentCore Gateway ターゲットへのアクセスを検証したりできます。また、エージェントから外部サービスや Gateway ターゲットへの安全なアウトバウンドアクセス（アウトバウンド認証）も提供します。既存の ID プロバイダー（Amazon Cognito など）と統合しながら、エージェントが独立して、または（OAuth を介して）ユーザーに代わって行動する際のアクセス許可の境界を適用します。

インバウンド認証は、AgentCore Runtime、AgentCore Gateway、または他の環境でホストされているかに関わらず、エージェントやツールを呼び出そうとする呼び出し元を検証します。インバウンド認証は IAM（SigV4 認証情報）または OAuth 認証と連携します。

デフォルトでは、Amazon Bedrock AgentCore は IAM 認証情報を使用します。つまり、エージェントへのユーザーリクエストはユーザーの IAM 認証情報で認証されます。OAuth を使用する場合は、AgentCore Runtime リソースまたは AgentCore Gateway エンドポイントを構成する際に以下を指定する必要があります：

- OAuth ディスカバリーサーバー URL — OpenID Connect ディスカバリー URL のパターン ^.+/\.well-known/openid-configuration$ に一致する必要がある文字列

- 許可されたオーディエンス — JWT トークンに許可されたオーディエンスのリスト

- 許可されたクライアント — 許可されたクライアント識別子のリスト

AgentCore CLI を使用する場合は、**configure** コマンドを使用する際に AgentCore Runtime の認証タイプ（および OAuth ディスカバリーサーバー）を指定できます。CreateAgentRuntime 操作と Amazon Bedrock AgentCore コンソールを使用することもできます。Gateway を作成する場合は、CreateGateway 操作またはコンソールを使用します。

ユーザーがエージェントを使用する前に、クライアントアプリケーションはユーザーを OAuth 認証機関で認証させる必要があります。クライアントはベアラートークンを受け取り、それを呼び出しリクエストでエージェントに渡します。受信時に、エージェントはアクセスを許可する前に認証サーバーでトークンを検証します。

## 概要

このチュートリアルでは、01-AgentCore-runtime でデプロイしたエージェントを変更し、ID プロバイダーとして Cognito を使用したインバウンド認証用に構成します。1 人のユーザーとアプリクライアントを持つ Cognito ユーザープールをセットアップします。Cognito ユーザープールを使用したインバウンド認証で Amazon Bedrock AgentCore Runtime を使用して既存のエージェントをホストする方法を学びます。

### チュートリアルのアーキテクチャ

<div style="text-align:center">
    <img src="images/inbound_auth_cognito.png" width="90%"/>
</div>

### チュートリアルの詳細

| 情報 | 詳細 |
|:--------------------|:---------------------------------------------------------------------------------|
| チュートリアルタイプ | 会話型 |
| エージェントタイプ | 単一 |
| エージェントフレームワーク | Strands Agents |
| LLM モデル | Anthropic Claude Sonnet 4 |
| チュートリアルコンポーネント | AgentCore Runtime でのエージェントのホスティング。Strands Agent と Amazon Bedrock モデルの使用 |
| チュートリアルの分野 | 分野横断的 |
| 例の複雑さ | 簡単 |
| インバウンド認証 | Cognito |
| 使用する SDK | Amazon BedrockAgentCore Python SDK と boto3 |

### チュートリアルの主な機能

* Amazon Cognito を使用したインバウンド認証による Amazon Bedrock AgentCore Runtime でのエージェントのホスティング
* Amazon Bedrock モデルの使用
* Strands Agents の使用

## 前提条件

このチュートリアルを実行するには以下が必要です：
* Python 3.10+
* AWS 認証情報
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker が稼働していること

In [ ]:
#!uv add -r requirements.txt --active

import os
os.environ['AWS_PROFILE'] = 'cline2'

### Cognito ユーザープールのプロビジョニング

Cognito ユーザープールを App クライアントと 1 人のテストユーザーでプロビジョニングしましょう。1/Cognito ディスカバリー URL と 2/Cognito アプリクライアント ID をメモしておいてください。これらは、エージェントを Cognito を使用したインバウンド認証用に構成する際に使用します。

注意: Cognito の access_token は 2 時間のみ有効です。access_token の有効期限が切れた場合は、setup_cognito_uer_pool.sh から **# Authenticate User** コマンドレットを実行して、別の access_token を取得できます。

In [ ]:
!chmod +x setup_cognito_user_pool.sh
!sh setup_cognito_user_pool.sh

## AgentCore Runtime へのエージェントのデプロイ準備

### Amazon Bedrock モデルを使用した Strands エージェント
01-AgentCore-runtime チュートリアルで作成した Strands エージェントから始めて、アイデンティティプロバイダーとして Amazon Cognito を使用するインバウンド認証を設定しましょう。

In [ ]:
%%writefile strands_claude.py
from strands import Agent, tool
from strands_tools import calculator # calculator ツールをインポート
import argparse
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel
import os

app = BedrockAgentCoreApp()

# カスタムツールの作成
@tool
def weather():
    """ 天気を取得 """ # ダミー実装
    return "sunny"


model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
# model_id = "us.anthropic.claude-sonnet-4-20250514-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)

@app.entrypoint
def strands_agent_bedrock(payload):
    """
    ペイロードでエージェントを呼び出す
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()


## AgentCore Runtime へのエージェントのデプロイ

`CreateAgentRuntime` 操作は包括的な構成オプションをサポートしており、コンテナイメージ、環境変数、暗号化設定を指定することができます。また、プロトコル設定 (HTTP、MCP) や認証メカニズムを構成して、クライアントがエージェントとどのように通信するかを制御することもできます。

**注意:** 運用のベストプラクティスは、コードをコンテナとしてパッケージ化し、CI/CD パイプラインと IaC を使用して ECR にプッシュすることです

このチュートリアルでは、Amazon Bedrock AgentCode Python SDK を使用して、アーティファクトを簡単にパッケージ化し、AgentCore ランタイムにデプロイします。

### ランタイムロールの作成

始める前に、AgentCore ランタイム用の IAM ロールを作成しましょう。これには、あなたのために事前に開発されたユーティリティ関数を使用します。

In [ ]:
import sys
import os

# 現在のノートブックのディレクトリを取得する
current_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in globals() else '.'))

# utils.py の場所に移動する
utils_dir = os.path.join(current_dir, '..')
utils_dir = os.path.abspath(utils_dir)

# Add to sys.path
sys.path.insert(0, utils_dir)

from utils import create_agentcore_role

agent_name="strands_claude"
agentcore_iam_role = create_agentcore_role(agent_name=agent_name)

### AgentCore Runtime デプロイメントの構成

次に、スターターツールキットを使用して、エントリーポイント、先ほど作成した実行ロール、および要件ファイルを含む AgentCore Runtime デプロイメントを構成します。また、起動時に Amazon ECR リポジトリを自動作成するようにスターターキットを構成します。

構成ステップ中に、アプリケーションコードに基づいて Docker ファイルが生成されます。

**重要** - 前のステップで取得した Cognito Discovery URL と Cognito App クライアント ID を更新してください。

In [ ]:
import boto3
import json
import os
import time

# リージョンを取得
region = boto3.session.Session().region_name
print(f"Using AWS Region: {region}")

# Cognito クライアントを初期化
cognito_client = boto3.client('cognito-idp', region_name=region)

# ユーザープールを一覧表示（DemoUserPoolを検索）
user_pools = cognito_client.list_user_pools(MaxResults=60)
pool_id = None
for pool in user_pools['UserPools']:
    if pool['Name'] == 'DemoUserPool':
        pool_id = pool['Id']
        break

if pool_id:
    print(f"Found User Pool ID: {pool_id}")
    
    # Discovery URLを構築
    discovery_url = f"https://cognito-idp.{region}.amazonaws.com/{pool_id}/.well-known/openid-configuration"
    print(f"Cognito Discovery URL: {discovery_url}")
    
    # クライアントIDを取得
    clients = cognito_client.list_user_pool_clients(
        UserPoolId=pool_id,
        MaxResults=60
    )
    
    client_id = None
    for client in clients['UserPoolClients']:
        if client['ClientName'] == 'DemoClient':
            client_id = client['ClientId']
            break
    
    if client_id:
        print(f"Cognito App Client ID: {client_id}")
        
        # アクセストークンを取得（ユーザー認証）
        try:
            auth_response = cognito_client.initiate_auth(
                ClientId=client_id,
                AuthFlow='USER_PASSWORD_AUTH',
                AuthParameters={
                    'USERNAME': 'testuser',
                    'PASSWORD': 'MyPassword123!'
                }
            )
            
            # アクセストークンを取得
            cognito_bearer_token = auth_response['AuthenticationResult']['AccessToken']
            token_preview = cognito_bearer_token[:20] + "..." if len(cognito_bearer_token) > 20 else cognito_bearer_token
            print(f"Cognito Bearer Token: {token_preview} (truncated)")
            
            # 変数に格納して後で使用
            print("\n以下の変数が設定されました：")
            print("discovery_url = 上記のCognito Discovery URL")
            print("client_id = 上記のCognito App Client ID")
            print("cognito_bearer_token = 上記のCognito Bearer Token")
            
            # 実際に変数を設定
            # これらの変数をノートブックの後続のセルで使用できます
            
            # 以下のコードをコメントアウトして、実際のノートブックのセルに追加してください
            # discovery_url = discovery_url
            # client_id = client_id
            # cognito_bearer_token = cognito_bearer_token
            
        except Exception as e:
            print(f"認証エラー: {str(e)}")
            print("注意: アクセストークンの有効期限が切れている可能性があります。")
            print("その場合は、setup_cognito_user_pool.shスクリプトを再実行してください。")
    else:
        print("DemoClient not found in the user pool.")
else:
    print("DemoUserPool not found. Please run the setup_cognito_user_pool.sh script first.")


In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name
region

#Cognito Discovery url を以下で更新してください。discovery url は「Cognito User Pool のプロビジョニング」セクションから取得できます
# discovery_url = '' 

#Cognito アプリクライアント ID を以下に更新してください。Cognito アプリクライアントは「Cognito ユーザープールのプロビジョニング」セクションから取得できます
# client_id = '' 

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    execution_role=agentcore_iam_role['Role']['Arn'],
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id]
        }
    }
)
response


## AgentCore の設定を確認する

In [ ]:
!head -n10 .bedrock_agentcore.yaml

### AgentCore Runtime へのエージェントの起動

Docker ファイルができたので、エージェントを AgentCore Runtime に起動しましょう。これにより Amazon ECR リポジトリと AgentCore Runtime が作成されます。

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [ ]:
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)
launch_result

### AgentCore ランタイムのステータス確認
AgentCore ランタイムをデプロイしたので、そのデプロイ状況を確認しましょう

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### AgentCore Runtime を認証なしで呼び出す

最後に、ペイロードを使って AgentCore Runtime を呼び出すことができます。以下のセルを実行してみると、**「AccessDeniedException: An error occurred (AccessDeniedException) when calling the InvokeAgentRuntime operation: Agent is configured for a different authorization token type」**（アクセス拒否例外：InvokeAgentRuntime 操作の呼び出し時にエラーが発生しました（アクセス拒否例外）：エージェントは異なる認証トークンタイプ用に構成されています）というエラーが表示されます。

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [ ]:
#invoke_response = agentcore_runtime.invoke({"prompt": "今の天気はどう?"})
#invoke_response

### AgentCore ランタイムを認証で呼び出す

正しい認証トークンタイプでエージェントを呼び出してみましょう。今回の場合、それは Cognito アクセストークンになります。「**Provision a Cognito User Pool**」のセルからアクセストークンをコピーしてください。

In [ ]:

#Cognito アクセストークンをここで更新してください。「Provision a Cognito User Pool」セルからアクセストークンをコピーしてください
# cognito_bearer_token=""

import boto3

cognito_client = boto3.client('cognito-idp', region_name="us-east-1")
auth_response = cognito_client.initiate_auth(
    ClientId=client_id,
    AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={
        'USERNAME': 'testuser',
        'PASSWORD': 'MyPassword123!'
    }
)

new_token = auth_response['AuthenticationResult']['AccessToken']
print(f"新しいトークン: {new_token[:20]}...")

invoke_response = agentcore_runtime.invoke({"prompt": "今の天気はどう?"}, bearer_token=new_token)
invoke_response


### AgentCore ランタイムを Python Http クライアントで呼び出す（オプション）

Boto3 は IAM sigv4 ベースのイングレスのみをサポートしています。そのため、ベアラートークンを使用してエージェントを呼び出すには、通常の Python HTTP クライアントを使用する必要があります。以下のサンプルでは、Python の request ライブラリを使用しています。

#### Agentcore の設定を確認しましょう

In [ ]:
!uv run debug_request.py

## クリーンアップ (任意)

作成した AgentCore Runtime をクリーンアップしましょう

In [ ]:
from boto3.session import Session
boto_session = Session()

agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)
ecr_client = boto3.client(
    'ecr',
    region_name=region
    
)

iam_client = boto3.client('iam')

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
    
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

policies = iam_client.list_role_policies(
    RoleName=agentcore_iam_role['Role']['RoleName'],
    MaxItems=100
)

for policy_name in policies['PolicyNames']:
    iam_client.delete_role_policy(
        RoleName=agentcore_iam_role['Role']['RoleName'],
        PolicyName=policy_name
    )
iam_response = iam_client.delete_role(
    RoleName=agentcore_iam_role['Role']['RoleName']
)

# おめでとうございます！